# Data analysis

In [1]:
import contextlib
import pandas as pd
import polars as pl
from tqdm import tqdm

# Loading the data

In [4]:
df = pl.read_parquet("df_gliner_full.parquet").unnest("res").explode("titulaire-nom-positions")
df.select(pl.selectors.by_name("titulaire-nom","titulaire-nom-positions","person","entities","political ideology"))

titulaire-nom,titulaire-nom-positions,person,entities,political ideology
str,i64,list[struct[2]],struct[7],str
"""Aulagne""",1240,"[{{""François MITTERRAND"",523,542},""right""}, {null,""right""}]","{[{""Bernard AULAGNE"",1232,1247}, {""François MITTERRAND"",523,542}],[],[],[],[],[{""RPR"",2273,2276}, {""UDF"",614,617}, … {""PS"",2283,2285}],[{""Ain"",1389,1392}, {""France"",2774,2780}]}","""right"""
"""Aulagne""",2866,"[{{""François MITTERRAND"",523,542},""right""}, {null,""right""}]","{[{""Bernard AULAGNE"",1232,1247}, {""François MITTERRAND"",523,542}],[],[],[],[],[{""RPR"",2273,2276}, {""UDF"",614,617}, … {""PS"",2283,2285}],[{""Ain"",1389,1392}, {""France"",2774,2780}]}","""right"""
"""Pujol""",208,"[{{""JUQUIN"",193,199},""left""}, {{""PUJOL"",208,213},""right""}]","{[{""JUQUIN"",1472,1478}, {""PUJOL"",208,213}],[{""GAUCHE"",3479,3485}, {""JUQUIN"",193,199}],[{""PUJOL"",208,213}, {""Jean-François MORTEL"",744,764}],[],[],[{""CEVIPOF"",1707,1714}, {""CEVIPOFSciences"",1680,1695}, … {""Sciences Po"",1660,1671}],[{""Afrique"",496,503}]}","""left"""
"""Pujol""",3567,"[{{""JUQUIN"",193,199},""left""}, {{""PUJOL"",208,213},""right""}]","{[{""JUQUIN"",1472,1478}, {""PUJOL"",208,213}],[{""GAUCHE"",3479,3485}, {""JUQUIN"",193,199}],[{""PUJOL"",208,213}, {""Jean-François MORTEL"",744,764}],[],[],[{""CEVIPOF"",1707,1714}, {""CEVIPOFSciences"",1680,1695}, … {""Sciences Po"",1660,1671}],[{""Afrique"",496,503}]}","""left"""
"""Boyon""",155,"[{{""Jacques BOYON"",147,160},""left""}, {{""Paul MORIN"",320,330},""left""}, {{""Jacques CHIRAC"",1745,1759},""right""}]","{[{""Paul MORIN"",4294,4304}, {""Jacques BOYON"",147,160}, {""Jacques CHIRAC"",1745,1759}],[],[{""Paul MORIN"",320,330}],[],[],[{""Assemblée Nationale"",807,826}, {""CEVIPOF"",20,27}, … {""Légion d'honneur"",273,289}],[{""BRESSE"",3598,3604}, {""BOURG"",3538,3543}, {""REVERMONT"",2998,3007}]}","""left"""
…,…,…,…,…
"""Virapoullé""",1360,"[{{""JEAN-PAUL VIRAPOULLE"",116,136},""right""}]","{[{""Jean-Paul VIRAPOULLE"",1350,1370}, {""MARC-LUC BOYER"",147,161}],[],[{""JEAN-PAUL VIRAPOULLE"",116,136}, {""Paul VERGES"",549,560}],[],[],[{""CEVIPOF"",360,367}],[{""Z.I. du Chaudron"",1445,1461}]}","""right"""
"""Virapoullé""",1425,"[{{""JEAN-PAUL VIRAPOULLE"",116,136},""right""}]","{[{""Jean-Paul VIRAPOULLE"",1350,1370}, {""MARC-LUC BOYER"",147,161}],[],[{""JEAN-PAUL VIRAPOULLE"",116,136}, {""Paul VERGES"",549,560}],[],[],[{""CEVIPOF"",360,367}],[{""Z.I. du Chaudron"",1445,1461}]}","""right"""
"""Vergès""",33,"[{{""Paul VERGÈS"",28,39},""right""}, {{""Jean-Paul Virapoullé"",595,615},""right""}, {{""Jean-Claude Fruteau"",1121,1140},""left""}]","{[{""Paul Vergès"",1400,1411}, {""Lucet LANGENIER"",3746,3761}, … {""Jean-Paul Virapoulé"",1670,1689}],[],[{""Jean-Paul Virapoulé"",1670,1689}, {""Jean-Paul Virapoullé"",595,615}, {""Jean-Claude Fruteau"",1121,1140}],[],[],[{""Assemblée Nationale"",1806,1825}, {""Sciences Po / fonds CEVIPOF"",0,27}],[{""Paris"",2035,2040}]}","""left"""


In [28]:
def position_is_inside_span(pos:int, span):
    # Returns a boolean
    # True if position is inside the span
    return (pos >= span["start"]) & (pos < span["end"])

df = df.unnest("entities").with_columns(
    pl.struct(["titulaire-nom-positions","PER"])
    .map_elements(lambda s: any([position_is_inside_span(s["titulaire-nom-positions"],span) for span in s["PER"]]), 
                  return_dtype=pl.Boolean)
    .alias("is-in-PER"),
    pl.struct(["titulaire-nom-positions","PER_LEFT"])
    .map_elements(lambda s: any([position_is_inside_span(s["titulaire-nom-positions"],span) for span in s["PER_LEFT"]]), 
                  return_dtype=pl.Boolean)
    .alias("is-in-PER_LEFT"),
    pl.struct(["titulaire-nom-positions","PER_RIGHT"])
    .map_elements(lambda s: any([position_is_inside_span(s["titulaire-nom-positions"],span) for span in s["PER_RIGHT"]]), 
                  return_dtype=pl.Boolean)
    .alias("is-in-PER_RIGHT"),
    pl.struct(["titulaire-nom-positions","PER_GAUCHE"])
    .map_elements(lambda s: any([position_is_inside_span(s["titulaire-nom-positions"],span) for span in s["PER_GAUCHE"]]), 
                  return_dtype=pl.Boolean)
    .alias("is-in-PER_GAUCHE"),
    pl.struct(["titulaire-nom-positions","PER_DROITE"])
    .map_elements(lambda s: any([position_is_inside_span(s["titulaire-nom-positions"],span) for span in s["PER_DROITE"]]), 
                  return_dtype=pl.Boolean)
    .alias("is-in-PER_DROITE"),
    pl.struct(["titulaire-nom-positions","ORG"])
    .map_elements(lambda s: any([position_is_inside_span(s["titulaire-nom-positions"],span) for span in s["ORG"]]), 
                  return_dtype=pl.Boolean)
    .alias("is-in-ORG"),
    pl.struct(["titulaire-nom-positions","LOC"])
    .map_elements(lambda s: any([position_is_inside_span(s["titulaire-nom-positions"],span) for span in s["LOC"]]), 
                  return_dtype=pl.Boolean)
    .alias("is-in-LOC")
)
# df.select(pl.selectors.by_name("titulaire-nom","titulaire-nom-positions","is-in-PER","PER","is-in-PER_LEFT"))


In [42]:
toto = df.get_column("is-in-PER").value_counts().rows_by_key("is-in-PER", unique=True)

In [58]:
df.with_columns(weight=1).select(pl.sum("is-in-PER","weight")
).with_columns(
    recall = pl.col("is-in-PER")/pl.col("weight")
).select(pl.selectors.by_name("is-in-PER","weight","recall"))

is-in-PER,weight,recall
u32,i32,f64
3283,9010,0.364373


In [53]:
df.with_columns(weight=1).group_by(
    pl.col("titulaire-nom"),maintain_order=True
).agg(
    pl.col("is-in-PER").sum().alias("toto"),
    pl.col("weight").sum().alias("total")
).with_columns(
    recall = pl.col("toto")/pl.col("total")
).select(pl.selectors.by_name("titulaire-nom","toto","total","recall"))

titulaire-nom,toto,total,recall
str,u32,i32,f64
"""Aulagne""",1,2,0.5
"""Pujol""",1,2,0.5
"""Boyon""",2,2,1.0
"""Saint-Pierre""",2,4,0.5
"""Mornet""",1,1,1.0
…,…,…,…
"""Poillot""",1,1,1.0
"""Delaporte""",1,1,1.0
"""Jégou""",0,2,0.0


In [8]:
df[0]["entities"]["PER"]

TypeError: cannot select elements using Sequence with elements of type 'str'